# Data inladen – GreatOutdoors SDM
Dit notebook laadt alle brondata (3x SQLite + 3x CSV) in het SDM.

**Cel 1** – Configuratie  
**Cel 2** – Helperfuncties  
**Cel 3** – `load()` – vul het SDM  
**Cel 4** – `clear()` – leeg het SDM  

In [1]:
# ──────────────────────────────────────────────
# CEL 1 – CONFIGURATIE
# Pas alleen hier de paden aan indien nodig
# ──────────────────────────────────────────────

import sqlite3
import pandas as pd
from pathlib import Path

SDM_PATH = "../data/sdm_deproject.db"

SQLITE_SOURCES = {
    "../GreatOutdoors/GreatOutdoors/GO_SALES-data.sqlite": [
        "order_header",
        "order_details",
        "order_method",
        "product_line",
        "product_type",
        "product",
        "sales_branch",
        "sales_staff",
        "retailer_site",
        "returned_item",
        "return_reason",
        "country",
    ],
    "../GreatOutdoors/GreatOutdoors/CRM-data.sqlite": [
        "customer",
        "customer_type",
        "customer_store",
        "customer_headquarters",
        "customer_segment",
        "customer_contact",
        "crm_country",
        "sales_territory",
        "age_group",
        "sales_demographic",
    ],
    "../GreatOutdoors/GreatOutdoors/GO_STAFF-data.sqlite": [
        "sales_representative",
        "sales_office",
        "course",
        "training",
        "satisfaction_type",
        "satisfaction",
    ],
}

CSV_SOURCES = {
    "../GreatOutdoors/GreatOutdoors/SALES_TARGET-data.csv":     "sales_target",
    "../GreatOutdoors/GreatOutdoors/PRODUCT_FORECAST-data.csv": "product_forecast",
    "../GreatOutdoors/GreatOutdoors/INVENTORY_LEVELS-data.csv": "inventory_levels",
}

print("Configuratie geladen")

Configuratie geladen


In [2]:
# ──────────────────────────────────────────────
# CEL 2 – HELPERFUNCTIES
# ──────────────────────────────────────────────

def _load_from_sqlite(sdm_con, source_path, tables):
    """Kopieer meerdere tabellen uit één SQLite-bronbestand naar het SDM."""
    if not Path(source_path).exists():
        print(f"  [SKIP] {source_path} niet gevonden")
        return
    src_con = sqlite3.connect(source_path)
    for table in tables:
        df = pd.read_sql(f"SELECT * FROM {table}", src_con)

        sdm_cols = [row[1] for row in sdm_con.execute(f"PRAGMA table_info({table})")]
        df.columns = df.columns.str.upper()
        sdm_cols_upper = [c.upper() for c in sdm_cols]
        df = df[[col for col in df.columns if col in sdm_cols_upper]]
        df.columns = [sdm_cols[sdm_cols_upper.index(col)] for col in df.columns]

        df.to_sql(table, sdm_con, if_exists="append", index=False)
        print(f"  [OK]   {table:35s} {len(df):>6} rijen  ←  {Path(source_path).name}")
    src_con.close()


def _load_from_csv(sdm_con, csv_path, table):
    """Laad één CSV-bestand in het SDM."""
    if not Path(csv_path).exists():
        print(f"  [SKIP] {csv_path} niet gevonden")
        return

    for encoding in ["utf-8", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(csv_path, encoding=encoding)
            break
        except UnicodeDecodeError:
            continue
    else:
        print(f"  [ERROR] {csv_path} kon niet gelezen worden (encoding onbekend)")
        return
    df = df.drop_duplicates()
    df.to_sql(table, sdm_con, if_exists="append", index=False)
    print(f"  [OK]   {table:35s} {len(df):>6} rijen  ←  {Path(csv_path).name}")


def _clear_table(sdm_con, table):
    """Verwijder alle rijen uit één SDM-tabel."""
    sdm_con.execute(f"DELETE FROM {table}")
    print(f"  [CLEAR] {table}")


def load():
    """Laad alle brondata in het SDM."""
    print("=== LADEN ===")
    sdm_con = sqlite3.connect(SDM_PATH)

    print("\n-- SQLite bronnen --")
    for source_path, tables in SQLITE_SOURCES.items():
        _load_from_sqlite(sdm_con, source_path, tables)

    print("\n-- CSV bronnen --")
    for csv_path, table in CSV_SOURCES.items():
        _load_from_csv(sdm_con, csv_path, table)

    sdm_con.commit()
    sdm_con.close()
    print("\n✓ Klaar – SDM gevuld.")


def clear():
    """Leeg alle SDM-tabellen (verwijder rijen, behoud structuur)."""
    print("=== LEEGMAKEN ===")
    sdm_con = sqlite3.connect(SDM_PATH)

    all_tables = [t for tables in SQLITE_SOURCES.values() for t in tables] + list(CSV_SOURCES.values())
    for table in all_tables:
        _clear_table(sdm_con, table)

    sdm_con.commit()
    sdm_con.close()
    print("\n✓ Klaar – SDM leeggemaakt.")


print("Functies gedefinieerd")

Functies gedefinieerd


In [3]:
# ──────────────────────────────────────────────
# CEL 3 – LAAD DATA IN HET SDM
# Voer deze cel uit om het SDM te vullen
# ──────────────────────────────────────────────

load()

=== LADEN ===

-- SQLite bronnen --


DatabaseError: Execution failed

In [ ]:
# ──────────────────────────────────────────────
# CEL 4 – LEEG HET SDM
# Voer deze cel uit om alle rijen te verwijderen
# ──────────────────────────────────────────────

clear()

=== LEEGMAKEN ===
  [CLEAR] order_header
  [CLEAR] order_details
  [CLEAR] order_method
  [CLEAR] product_line
  [CLEAR] product_type
  [CLEAR] product
  [CLEAR] sales_branch
  [CLEAR] sales_staff
  [CLEAR] retailer_site
  [CLEAR] returned_item
  [CLEAR] return_reason
  [CLEAR] country
  [CLEAR] customer
  [CLEAR] customer_type
  [CLEAR] customer_store
  [CLEAR] customer_headquarters
  [CLEAR] customer_segment
  [CLEAR] customer_contact
  [CLEAR] crm_country
  [CLEAR] sales_territory
  [CLEAR] age_group
  [CLEAR] sales_demographic
  [CLEAR] sales_representative
  [CLEAR] sales_office
  [CLEAR] course
  [CLEAR] training
  [CLEAR] satisfaction_type
  [CLEAR] satisfaction
  [CLEAR] sales_target
  [CLEAR] product_forecast
  [CLEAR] inventory_levels

✓ Klaar – SDM leeggemaakt.
